# Structured outputs — schema-valid data from Claude

Structured outputs make Claude return data that conforms to a schema, so you can use it directly
instead of parsing prose. The recommended path is `client.messages.parse(...)` with a **Pydantic
model** — the SDK enforces the schema and returns a validated object. You can also drop to the
raw `output_config.format` (JSON schema), or use `strict: True` on a tool for guaranteed-valid
tool arguments.

Here we extract structured wildlife **sighting reports** from messy free-text field notes.

**Requirements:** `ANTHROPIC_API_KEY` (env var or a `.env` at the repo root).

## Setup

The Pydantic model, prompt, and helpers live in `_structured_outputs.py`.

In [ ]:
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

for _p in (".", "structured_outputs"):
    if os.path.isfile(os.path.join(_p, "_structured_outputs.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _structured_outputs import (
    EXTRACT_PROMPT,
    MODEL,
    SAMPLE_NOTES,
    SIGHTING_JSON_SCHEMA,
    SightingReport,
    extract,
    extract_many,
)

load_dotenv()
client = Anthropic()

## The schema

A `SightingReport` Pydantic model defines exactly the shape we want — typed fields, an enum for
time of day, and a bounded confidence. This is the contract Claude must satisfy.

In [ ]:
print(SightingReport.model_fields.keys())
SAMPLE_NOTES[0]

## Extract one note (typed)

`extract` calls `client.messages.parse(..., output_format=SightingReport)` and returns a
**validated `SightingReport` instance** — no JSON parsing, no missing-key surprises.

In [ ]:
report = extract(client, SAMPLE_NOTES[0])
print(type(report).__name__)
print("species   :", report.species, "/", report.scientific_name)
print("count     :", report.count)
print("time_of_day:", report.time_of_day.value)
print("behaviors :", report.behaviors)
print("confidence:", report.confidence)

## Batch → a clean table

Because every result is the same shape, a list of notes becomes a tidy table of records.

In [ ]:
reports = extract_many(client, SAMPLE_NOTES)
rows = [r.model_dump() for r in reports]
for row in rows:
    print(f"{row['species']:<12} n={row['count']} {row['time_of_day']:<7} conf={row['confidence']}  {row['behaviors']}")

## The raw form: `output_config.format`

`messages.parse` builds on `output_config.format` with a JSON schema. You can use it directly when
you don't want Pydantic. Note the strict validator **doesn't support numeric `minimum`/`maximum`** —
express bounded integers with an `enum` (as below) and require `additionalProperties: false`.

In [ ]:
import json

resp = client.messages.create(
    model=MODEL, max_tokens=256,
    messages=[{"role": "user", "content": EXTRACT_PROMPT + SAMPLE_NOTES[2]}],
    output_config={"format": SIGHTING_JSON_SCHEMA},
)
text = next(b.text for b in resp.content if b.type == "text")
print(json.loads(text))

## Strict tool use

For *tool* arguments (not the final answer), add `"strict": True` to the tool's `input_schema` so
Claude's `tool_use.input` is guaranteed schema-valid — handy when you'll pass those arguments
straight into a function or API.

In [ ]:
log_tool = {
    "name": "log_sighting",
    "description": "Record a wildlife sighting in the database.",
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "species": {"type": "string"},
            "count": {"type": "integer"},
            "time_of_day": {"type": "string", "enum": ["dawn", "day", "dusk", "night", "unknown"]},
        },
        "required": ["species", "count", "time_of_day"],
        "additionalProperties": False,
    },
}

resp = client.messages.create(
    model=MODEL, max_tokens=512, tools=[log_tool],
    tool_choice={"type": "tool", "name": "log_sighting"},
    messages=[{"role": "user", "content": "Log this: " + SAMPLE_NOTES[1]}],
)
print(next(b.input for b in resp.content if b.type == "tool_use"))

## Notes

- **`messages.parse` is the easy path** — pass a Pydantic model as `output_format`, read
  `response.parsed_output`. Use the raw `output_config.format` when you don't want Pydantic, and
  `strict: True` tools when it's *tool arguments* you need validated.
- **Strict schema limits:** require `additionalProperties: false`, list `required` fields, and use
  `enum` instead of numeric `minimum`/`maximum`.
- **Incompatible with citations.** Enabling structured outputs (`output_config.format`) together
  with citations returns a 400 — pick one. (See the `citations/` topic.)
- **Great for extraction/classification pipelines** where downstream code needs reliable shapes.